# Sandboxed instances

Every code cell has a **Sandbox** button when the running build has the
`dv_` instance ABI (5.5.1_build3 and later — it is hidden otherwise).

It runs the same source as a Diluvium *instance* rather than in the
notebook's Lua state. That is the difference between a notebook and a
lab: not a second way to print things, but a way to watch a program
**cost** something.

In [ ]:
print("instances available:", ...)
-- (this cell is here so you have a Sandbox button to press;
--  look at the toolbar above it)

## Try it

Press **Sandbox** on the cell below, then **Run sandboxed**. Compare
the panel with what Ctrl+Enter gives you.

In [ ]:
local total = 0
for i = 1, 200000 do total = total + i end
print("total", total)

The panel reports:

- **instructions used, against the budget** — a real count from the VM
- **peak memory** — the high-water mark, not the current figure
- **its queues**, and how full they are
- whether it finished, errored, ran out of budget, or parked

## It shares nothing with the notebook

Its own state, its own globals, its own queues. Run the first cell
normally, then **Sandbox** the second.

In [ ]:
marker = "set by the notebook"
print(marker)

In [ ]:
print("what the sandbox sees:", tostring(marker))

## The budget is the point

A cell that loops forever costs you a worker and a **Stop**, which
takes every variable with it. The same loop in an instance costs 200,000
instructions and reports why it stopped.

Press **Sandbox**, set the budget to *200 thousand*, and run it. Then
check the kernel is still fine — it is.

(Do **not** press Ctrl+Enter on this one.)

In [ ]:
while true do end

In [ ]:
-- the notebook kernel, entirely unharmed
print(6 * 7)

### Why there is always a budget

Because **the instruction counter *is* the budget hook**. An instance
run with no budget reports zero instructions however much work it did —
measuring at all costs an armed hook. So the panel always sets one, and
the control chooses how big.

## Parking

`queue.wait` cannot work in a notebook cell: parking means yielding, and
something has to resume it. In an instance it works — and the panel
reports the program as parked, with what it is waiting on.

**Sandbox** this one.

In [ ]:
local work = queue.declare("work", { capacity = 4 })
print("declared, now parking")
queue.wait({ work })
print("this line is never reached here")

Nothing in the Lab can send it a message. That is a host loop's job,
and the swarm layer it belongs to is not in the browser artifact — so
the panel says so rather than leaving you to wonder.

The queue table below the report shows `work` sitting at `0 / 4`.

## Queues, seen from outside

The panel reads the instance's queues through the host ABI after the
run — which is what a supervisor would do. Push something and watch
`outbox` fill up.

In [ ]:
local out = queue.lookup("outbox")
for i = 1, 3 do
  queue.push(out, { seq = i, note = "for the host" })
end
print("pushed 3; the queue table below will show outbox at 3 / 64")

## What a sandboxed run refuses

The Lab creates instances with `DV_FLAG_TEXT_ONLY`, so a sandboxed run
accepts **source only** and refuses precompiled chunks. That is the
right default for code you pasted: the loader checks an instruction's
operands against the prototype that owns them, but Lua 5.1 shipped a
fuller checker than that and still had escapes. Source-only removes the
question rather than answering it.

## What it is not

An instance is an isolation and accounting boundary, not a security
boundary — the capability layer is a structuring device while `debug`
is available to guests, which upstream states plainly and which is part
of why `build3` is marked a prerelease.

Treat a program you run here as one you wrote or templated. That is
already true of every cell in this notebook, so the sandbox does not
make it *less* safe — it just does not make it safe to run a stranger's
code either.